# MotorCam – fáze 2: model povrchu silnice
Transfer learning na **MobileNetV3-Small** (volitelně EfficientNet-B0) v PyTorch.
Augmentace: jas/kontrast, stíny, rozmazání pohybem, déšť (`src/models/augment.py`).
Výstupy: accuracy, F1 po třídách, confusion matrix → `results/` na Google Drive.

**Runtime → Change runtime type → T4 GPU.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/motorcam'
REPO_URL = 'https://github.com/MiniGue1/motorcam.git'   # u soukromého repa: https://<token>@github.com/...

import os
%cd /content
if not os.path.exists('motorcam'):
    !git clone -q $REPO_URL motorcam
%cd /content/motorcam
!pip install -q -r requirements.txt
# archivy, vlastní data, modely a výsledky trvale na Google Drive
for d in ['data/raw/rdd2022/archives', 'data/raw/rscd/archives', 'data/custom', 'models', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True); os.makedirs(os.path.dirname(d) or '.', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(f'{DRIVE_DIR}/{d}', d)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENÍ – Runtime → Change runtime type → T4 GPU')

In [ ]:
# RSCD z fáze 1 (archiv na Drive), vlastní anotace z Label Studia
RSCD_ARCHIVE = f'{DRIVE_DIR}/data/raw/rscd/archives/rscd.zip'   # uprav podle názvu
!python -m src.dataset.download rscd --archive "$RSCD_ARCHIVE"
!python -m src.dataset.rscd_to_classes
import os
if os.path.exists('data/custom/labelstudio_export.json'):
    !python -m src.dataset.import_annotations labelstudio data/custom/labelstudio_export.json

## Ukázka augmentací (obrázek do práce)

In [ ]:
import glob
from IPython.display import Image, display
sample = sorted(glob.glob('data/processed/rscd_cls/train/sterk/*'))[0]
!python -m src.models.augment_preview "$sample"
display(Image('results/figures/augmentace.png'))

## Trénink na veřejných datech (RSCD)
~6 000 snímků na třídu, 25 epoch ≈ 20–30 min na T4. Nejlepší model podle F1 na validaci.

In [ ]:
!python -m src.models.train_surface --data data/processed/rscd_cls --name surface_public \
    --arch mobilenet_v3_small --epochs 25 --batch 128 --lr 1e-3 --workers 2

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/surface_public_uceni.png'))
display(Image('results/figures/surface_public_test_confusion.png'))
import pandas as pd; pd.read_csv('results/tables/surface_public_test_metriky.csv')

## Volitelně: EfficientNet-B0 pro srovnání architektur
Přesnější, ale pomalejší na telefonu – do práce se hodí tabulka přesnost vs. rychlost.

In [ ]:
# !python -m src.models.train_surface --data data/processed/rscd_cls --name surface_effb0 --arch efficientnet_b0 --epochs 20 --batch 96

## Doladění na vlastních (českých) záběrech
Start z modelu na veřejných datech, menší learning rate. Testovací jízdy se do tréninku nedostanou
(dělení po videích v `import_annotations`).

In [ ]:
import os
if os.path.exists('data/processed/custom_cls/train'):
    !python -m src.models.train_surface --data data/processed/custom_cls data/processed/rscd_cls \
        --init models/surface_public/best.pt --name surface_finetuned --epochs 12 --lr 3e-4 --batch 64
else:
    print('Zatím nejsou vlastní anotace (data/processed/custom_cls) – nejdřív fáze 1, bod 4.')